In [23]:
import pandas as pd

df = pd.read_csv("C:/Users/gromm/Downloads/ResidencesPrincipales.csv")
display(df)

,statut_occupation,2001,2004,2007,2010,2013,2016,2019,2022,2025,2028,2031,2034
0,Propriétaires,55.90,56.60,57.20,57.50,57.70,57.70,57.70,NaN,NaN,NaN,NaN,NaN
1,Non accédants,34.90,36.40,37.70,37.80,37.90,37.80,37.70,NaN,NaN,NaN,NaN,NaN
2,Accédants,21.00,20.20,19.60,19.70,19.80,19.90,20.00,NaN,NaN,NaN,NaN,NaN
3,Locataires,39.70,39.40,39.30,39.40,39.50,39.80,39.90,NaN,NaN,NaN,NaN,NaN
4,Bailleurs publics,17.90,17.70,17.50,17.30,17.20,17.10,17.00,NaN,NaN,NaN,NaN,NaN
5,Bailleurs privés,21.80,21.70,21.80,22.10,22.30,22.70,22.90,NaN,NaN,NaN,NaN,NaN
6,Autres statuts1,4.40,4.00,3.50,3.10,2.80,2.50,2.40,NaN,NaN,NaN,NaN,NaN
7,Total des résidences principales (en milliers),"24,973.00","26,016.00","26,993.00","27,786.00","28,516.00","29,237.00","29,916.00",NaN,NaN,NaN,NaN,NaN


In [ ]:
# Convert year columns in numerics

for col in df.columns[1:]:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace(" ", "")
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Melt the dataset to fit the LinearRegression process

df_long = df.melt(
    id_vars="statut_occupation",
    var_name="annee",
    value_name="valeur"
)

df_long["annee"] = df_long["annee"].astype(int)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Create Dataframe for years

annees_futures = pd.DataFrame({
    "annee": [2022, 2025, 2028, 2031, 2034]
})

predictions = []

# For loop on every columns 

for categorie in df_long["statut_occupation"].unique():
    
    data = df_long[df_long["statut_occupation"] == categorie].dropna()
    
    X = data[["annee"]]
    y = data["valeur"]
    
    model = LinearRegression()
    model.fit(X, y)
    
    # Prediction for each columns 

    pred = model.predict(annees_futures)
    
    temp = annees_futures.copy()
    temp["statut_occupation"] = categorie
    temp["prediction"] = pred
    
    # Add to temporary dataframe

    predictions.append(temp)

# Create exploitable dataframe

predictions_df = pd.concat(predictions)

In [ ]:
pred_table = predictions_df.pivot(
    index="statut_occupation",
    columns="annee",
    values="prediction"
)

display(pred_table)

annee,2022,2025,2028,2031,2034
statut_occupation,,,,,
Accédants,19.542857,19.421429,19.300000,19.178571,19.057143
Autres statuts1,1.857143,1.510714,1.164286,0.817857,0.471429
Bailleurs privés,23.014286,23.221429,23.428571,23.635714,23.842857
Bailleurs publics,16.785714,16.635714,16.485714,16.335714,16.185714
Locataires,39.800000,39.857143,39.914286,39.971429,40.028571
Non accédants,38.800000,39.207143,39.614286,40.021429,40.428571
Propriétaires,58.342857,58.632143,58.921429,59.210714,59.500000
Total des résidences principales (en milliers),30890.142857,31704.214286,32518.285714,33332.357143,34146.428571


In [ ]:
# Merge pred_table into df

df_merge = pd.merge(df, pred_table, how = "left", on = 'statut_occupation')

display(df_merge)

,statut_occupation,2001,2004,2007,2010,2013,2016,2019,2022,2025,2028,2031,2034,2022,2025,2028,2031,2034
0,Propriétaires,55.9,56.6,57.2,57.5,57.7,57.7,57.7,NaN,NaN,NaN,NaN,NaN,58.342857,58.632143,58.921429,59.210714,59.500000
1,Non accédants,34.9,36.4,37.7,37.8,37.9,37.8,37.7,NaN,NaN,NaN,NaN,NaN,38.800000,39.207143,39.614286,40.021429,40.428571
2,Accédants,21.0,20.2,19.6,19.7,19.8,19.9,20.0,NaN,NaN,NaN,NaN,NaN,19.542857,19.421429,19.300000,19.178571,19.057143
3,Locataires,39.7,39.4,39.3,39.4,39.5,39.8,39.9,NaN,NaN,NaN,NaN,NaN,39.800000,39.857143,39.914286,39.971429,40.028571
4,Bailleurs publics,17.9,17.7,17.5,17.3,17.2,17.1,17.0,NaN,NaN,NaN,NaN,NaN,16.785714,16.635714,16.485714,16.335714,16.185714
5,Bailleurs privés,21.8,21.7,21.8,22.1,22.3,22.7,22.9,NaN,NaN,NaN,NaN,NaN,23.014286,23.221429,23.428571,23.635714,23.842857
6,Autres statuts1,4.4,4.0,3.5,3.1,2.8,2.5,2.4,NaN,NaN,NaN,NaN,NaN,1.857143,1.510714,1.164286,0.817857,0.471429
7,Total des résidences principales (en milliers),24973.0,26016.0,26993.0,27786.0,28516.0,29237.0,29916.0,NaN,NaN,NaN,NaN,NaN,30890.142857,31704.214286,32518.285714,33332.357143,34146.428571


In [ ]:
# Removal of columns with NaN values

df_merge.dropna(axis = 1, inplace = True)

display(df_merge)

,statut_occupation,2001,2004,2007,2010,2013,2016,2019,2022,2025,2028,2031,2034
0,Propriétaires,55.9,56.6,57.2,57.5,57.7,57.7,57.7,58.342857,58.632143,58.921429,59.210714,59.500000
1,Non accédants,34.9,36.4,37.7,37.8,37.9,37.8,37.7,38.800000,39.207143,39.614286,40.021429,40.428571
2,Accédants,21.0,20.2,19.6,19.7,19.8,19.9,20.0,19.542857,19.421429,19.300000,19.178571,19.057143
3,Locataires,39.7,39.4,39.3,39.4,39.5,39.8,39.9,39.800000,39.857143,39.914286,39.971429,40.028571
4,Bailleurs publics,17.9,17.7,17.5,17.3,17.2,17.1,17.0,16.785714,16.635714,16.485714,16.335714,16.185714
5,Bailleurs privés,21.8,21.7,21.8,22.1,22.3,22.7,22.9,23.014286,23.221429,23.428571,23.635714,23.842857
6,Autres statuts1,4.4,4.0,3.5,3.1,2.8,2.5,2.4,1.857143,1.510714,1.164286,0.817857,0.471429
7,Total des résidences principales (en milliers),24973.0,26016.0,26993.0,27786.0,28516.0,29237.0,29916.0,30890.142857,31704.214286,32518.285714,33332.357143,34146.428571


In [36]:
df_merge.to_csv("ResidencesPrincipales_Exploitable.csv", index = False)